# Cloud Deployment & DevOps
This notebook covers cloud service models, CI/CD pipelines, GitHub Actions and GitLab CI, container orchestration, Kubernetes basics, and GPU scheduling in Kubernetes.

**Focus:** Deploying and scaling data science systems.

> This week builds directly on **Week 5**: we take the Flask API you Dockerized last week and do the following:
-  (1) put it behind an automated CI/CD pipeline, 
-  (2) then, deploy it onto a Kubernetes cluster **instead of running `docker run` by hand**.
---
## **1. Cloud Service Models (IaaS, PaaS, SaaS)**

**Concepts:**
- Cloud computing is usually described as a spectrum of *how much you manage yourself* vs *how much the provider manages for you*.
- The three classic service models:
  1. **IaaS (Infrastructure as a Service)**  &rarr;  you rent raw compute, storage, and networking (e.g. Amazon EC2- AWS EC2 (Elastic Compute Cloud), Google Compute Engine). You install and manage the OS, runtime, and app yourself.
  2. **PaaS (Platform as a Service)**  &rarr;  the provider manages the OS and runtime; you just deploy code (e.g. Heroku, Google App Engine, AWS Elastic Beanstalk).
  3. **SaaS (Software as a Service)**  &rarr;  you use a finished application over the internet and manage nothing (e.g. Gmail, Slack, Notion).

**Who manages what:**

| Layer | On-Premises | IaaS | PaaS | SaaS |
|---|---|---|---|---|
| Application | You | You | You | Provider |
| Data | You | You | You | Provider |
| Runtime | You | You | Provider | Provider |
| Middleware | You | You | Provider | Provider |
| Operating System | You | You | Provider | Provider |
| Virtualization | You | Provider | Provider | Provider |
| Servers/Storage/Networking | You | Provider | Provider | Provider |

 **Where does Kubernetes fit?** 
- A managed Kubernetes service (e.g. Google Kubernetes Engine (GKE), Amazon Elastic Kubernetes Service (EKS)) sits between IaaS and PaaS: the provider manages the control plane (the "brain" of the cluster),
- while you still define and manage what runs inside it  &rarr;  which is exactly what we'll be doing later in this notebook.

**Questions** [Don't forget to answer these reflection questions]
- Where would deploying your Week 5 Dockerized Flask app straight onto a rented AWS EC2 virtual machine sit on this spectrum? Why?
- Why might a data science team choose PaaS for a quick prototype API, but IaaS/Kubernetes for a production model-serving system with strict latency and scaling needs?

## **2. CI/CD Pipelines**

**Concepts:**
- **Continuous Integration (CI):** every time code is pushed, it is automatically built and tested, catching bugs before they reach other developers.
- **Continuous Delivery (CD):** every change that passes CI is automatically packaged into a release that *could* be deployed at any time (a human still clicks "deploy").
- **Continuous Deployment (also CD):** goes one step further  &rarr;  every change that passes CI is deployed to production **automatically**, with no human step in between.

**A typical pipeline:**
```
 ┌────────┐    ┌───────┐    ┌─────────┐    ┌────────┐    ┌────────┐
 │  Code  │ -> │ Build │ -> │  Test   │ -> │Package │ -> │ Deploy │
 │  push  │    │       │    │(pytest) │    │(Docker)│    │ (k8s)  │
 └────────┘    └───────┘    └─────────┘    └────────┘    └────────┘
```

**Why it matters for data science systems specifically:**
- Model-serving code changes just as often as any other software  &rarr;  CI/CD prevents a broken preprocessing step from silently reaching production.
- Combined with Docker (Week 5), CI/CD guarantees the *exact same image* that passed tests is the one that gets deployed  &rarr;  no "it worked on my machine."

**Questions** [Don't forget to answer these reflection questions]
- What is the practical difference between Continuous Delivery and Continuous Deployment? Which would you choose for a model that retrains nightly and needs a human to sanity-check accuracy before going live?
- In the pipeline diagram above, which stage would catch a bug where `/health` starts returning a 500 error?

## **3. GitHub Actions**

**Concepts:**
- A **workflow** is an automated process defined in a YAML file under `.github/workflows/`.
- A workflow is made of **jobs**; each job runs on a fresh virtual machine called a **runner**.
- Each job is made of **steps**, executed in order (checkout code, set up Python, install dependencies, run tests, ...).
- Workflows are launched by **triggers**  &rarr;  e.g. `on: push`, `on: pull_request`, or a schedule.

### Instructions
1. Navigate to the **Week6 directory**.
2. Open [`.github/workflows/ci.yml`](.github/workflows/ci.yml) and read it top to bottom alongside the concepts above.
3. Note how each YAML key maps onto a concept: `on:` = trigger, `jobs:` = jobs, `runs-on:` = which runner, `steps:` = the ordered list of commands.
4. This workflow runs [`test_app.py`](test_app.py) against [`app.py`](app.py) using `pytest`. Open both files and check you understand what's being tested.
5. To actually see this run, you need a GitHub repository:
   - Initialize git in the **Week6 directory**: `git init`
   - Create a new **empty** repository on GitHub (no README).
   - `git remote add origin <your-repo-url>`
   - `git branch -M main`
   - `git add .`
   - `git commit -m "Week 6: add CI workflow"`
   - `git push -u origin main`
6. Open the **Actions** tab on your GitHub repository page. You should see the `CI` workflow running (or already completed with a green check ✅).

> **No GitHub account / offline?** You can still validate the pipeline logic locally before pushing anywhere  &rarr;  run the exact same commands the workflow runs:
```bash
pip install -r requirements.txt
pytest -v
```
> If these pass locally, the GitHub Actions job will pass too, since it runs the identical commands on a clean machine.

**Questions** [Don't forget to answer these reflection questions]
- What would happen to the "green check" on a pull request if you deliberately broke `test_app.py` and pushed that change? Why is this useful even when working alone?
- Why does the workflow install dependencies with `pip install -r requirements.txt` on *every single run*, instead of reusing a previously-installed environment?
- The workflow triggers `on: push` and `on: pull_request` to `main`. Why might a team deliberately *not* want it to trigger on every branch?

## **4. GitLab CI Concepts**

GitLab CI achieves the same goal as GitHub Actions  &rarr;  automatically build and test code on every push  &rarr;  but with a slightly different vocabulary and file layout.

**Side-by-side comparison:**

| Concept | GitHub Actions | GitLab CI |
|---|---|---|
| Config file location | `.github/workflows/*.yml` | `.gitlab-ci.yml` (repo root) |
| Trigger | `on:` | implicit (runs on every push) + `rules:` to restrict |
| Grouping | `jobs:` | `stages:` + `jobs:` |
| Execution environment | `runs-on:` (a runner) | `image:` (a Docker image) |
| Ordered commands | `steps:` (list of named actions/commands) | `script:` (flat list of shell commands) |
| Setup commands | separate `steps` | `before_script:` |

### Instructions
1. Open [`.gitlab-ci.yml`](.gitlab-ci.yml) in the **Week6 directory**.
2. Compare it line-by-line against [`.github/workflows/ci.yml`](.github/workflows/ci.yml) using the table above as a guide.
3. Both files do the same job: install dependencies, then run `pytest -v`.

> We are not pushing to GitLab in this practical lecture  &rarr;  the goal is to be able to **read** either syntax, since real-world teams use both depending on where their repository is hosted.

**Questions** [Don't forget to answer these reflection questions]
- In `.gitlab-ci.yml`, what does `stage: test` mean, and what would you add to run a *second* stage (e.g. `deploy`) only after `test` succeeds?
- If you moved this project from GitHub to GitLab tomorrow, which file(s) would you delete, and which would you keep?

## **5. Container Orchestration**

**Concepts:**
- In Week 5 you ran **one** container by hand with `docker run`. That's fine for one container, but real systems typically need many containers (an API, a database, a cache, a model server...) that must be started, networked together, restarted on failure, and scaled up or down.
- **Container orchestration** is the general term for tools that manage this automatically. Problems it solves:
  - **Scheduling**  &rarr;  deciding which machine runs which container.
  - **Scaling**  &rarr;  adding/removing container instances based on load.
  - **Self-healing**  &rarr;  automatically restarting or replacing crashed containers.
  - **Service discovery & load balancing**  &rarr;  letting containers find and talk to each other, and spreading traffic across replicas.
  - **Rolling updates**  &rarr;  deploying a new version with zero downtime.
- **Docker Compose** is a lightweight, single-machine step toward orchestration: it lets you define multiple related containers in one YAML file. **Kubernetes** (next section) is the industry-standard orchestrator for running containers across a whole *cluster* of machines.

### Instructions (Docker Compose warm-up)
1. Navigate to the **Week6 directory** and open [`docker-compose.yml`](docker-compose.yml).
2. Notice it defines **two** services (`web` and `redis`) in a single file  &rarr;  compare this to the single `docker run` command from Week 5.
3. In your terminal: `docker compose up --build` 
> Builds the Docker images (if needed) and then creates and starts all the services defined in the docker-compose.yml file. The --build option ensures that any changes to the application code, dependencies, or Dockerfile are included before the containers are launched.
4. Navigate to [http://localhost:5000/](http://localhost:5000/)  &rarr;  same Flask app as Week 5, now started as part of a multi-container stack.
5. In a second terminal, run `docker ps`  &rarr;  you should see **two** running containers.
6. Stop everything with `docker compose down`.

**Questions** [Don't forget to answer these reflection questions]
- What is the smallest change you'd need to make to `docker-compose.yml` to run **three** replicas of the `web` service? (Hint: search "docker compose scale".) Would this alone be a good way to run production traffic? Why might a real orchestrator like Kubernetes still be needed?
- Why is `restart: unless-stopped` in `docker-compose.yml` a small preview of what Kubernetes calls "self-healing"?

## **6. Kubernetes Basics**

Docker Compose (Section 5) showed you multi-container coordination on **one machine**. Kubernetes takes the same idea, declare what you want running and let a controller keep it that way, and scales it across a **cluster** of machines.

> Everything in this section builds toward one concrete goal: taking the same Flask container from Week 5 and running it in a way that survives crashes without you touching it.

**Key Concepts:**

| Term | Description |
|---|---|
| **Cluster** | A set of machines (nodes) that run your containers, managed as one unit. |
| **Node** | A single machine (physical or virtual) in the cluster. |
| **Pod** | The smallest deployable unit in Kubernetes: one or more tightly-coupled containers that share networking and storage. Usually one container per Pod. |
| **Deployment** | A controller that manages a set of identical Pods, keeping the desired number running and handling rolling updates. |
| **ReplicaSet** | What a Deployment uses internally to keep the right *number* of Pod replicas alive. You rarely create these directly. |
| **Service** | A stable network endpoint that load-balances traffic across a set of Pods, even as individual Pods are replaced. |
| **Namespace** | A way to logically partition a cluster (e.g. `dev`, `staging`, `prod`). |

**Why does Kubernetes need both a Pod *and* a Deployment? Why not just run containers directly?**

Because a lone / standalone Pod has no memory. If you start one Pod by hand and it crashes, nothing brings it back. It's gone, same as a `docker run` container that dies. 

- A **Deployment** exists to describe *desired state*: "I always want 2 healthy copies of this Pod running."
- Kubernetes then runs a continuous **control loop**, a background process that repeatedly asks "does what's actually running match what was requested?" and takes corrective action the moment it doesn't. That loop is what turns "start some containers" into "guarantee this many containers are always running." 
- It's the same self-healing and scaling behavior you'll trigger by hand in Step 3, except normally nobody has to trigger it; it just happens.

Concretely, when you write `replicas: 2` in a Deployment, you're not issuing a one-time command like `docker run`. You're registering a standing instruction that Kubernetes rechecks forever, until you change or delete it.

### Installation & Verification

- Install a local cluster tool: [minikube](https://minikube.sigs.k8s.io/docs/start/) (or [kind](https://kind.sigs.k8s.io/) as an alternative). A real Kubernetes cluster is normally a fleet of cloud machines. minikube runs a single-node cluster inside a VM/container on your laptop, so you can learn the exact same `kubectl` workflow without needing a cloud account.
- Install `kubectl` for [Windows](https://kubernetes.io/docs/tasks/tools/install-kubectl-windows/) / [macOS](https://kubernetes.io/docs/tasks/tools/install-kubectl-macos/) / [Linux](https://kubernetes.io/docs/tasks/tools/install-kubectl-linux/). minikube runs the cluster, but `kubectl` is the command-line client you actually talk to it with. Almost everything below is a `kubectl` command.
- Verify: `minikube version` and `kubectl version --client`
- **Start your local cluster:** `minikube start`. This supplies the single-node cluster and configures `kubectl` to point at it.

### General kubectl Commands (Check them well)

| Command | Description | When you'd reach for it |
|---|---|---|
| `kubectl get pods` | Lists all Pods in the current namespace. | Quick "is it up?" check. |
| `kubectl get deployments` | Lists all Deployments and how many replicas are ready. | Confirms desired vs. actual replica count. |
| `kubectl get services` | Lists all Services and their exposed ports. | Confirms how to reach your app. |
| `kubectl describe pod <pod-name>` | Shows detailed status/events for one Pod. | The first place to look when a Pod won't start. It shows the actual error (image pull failure, crash loop, failed probe, etc.), which `kubectl get pods` won't show you. |
| `kubectl logs <pod-name>` | Shows stdout/stderr logs of a Pod, same idea as `docker logs`. | Debugging application-level errors once the Pod is running. |
| `kubectl delete -f <file>.yaml` | Removes whatever resources were created from that manifest. | Tearing down what you applied, cleanly. |

`describe` vs. `logs` is a distinction worth internalizing now: `describe` tells you why Kubernetes couldn't schedule or start something; `logs` tells you what your application printed once it did start. A Pod stuck in `Pending` needs `describe`. A Pod that's `Running` but returning errors needs `logs`.

---

### Step 1: Build the image Kubernetes will run

> **Mac/Linux:**
```bash
eval $(minikube docker-env)      # Mac/Linux, makes `docker build` build INTO minikube
docker build -t flask-week6:latest .
```
> **Windows (PowerShell):** 
```bash
minikube docker-env | Invoke-Expression instead of the `eval` line.
docker build -t flask-week6:latest .
```

**Why this step exists:** minikube runs its **own isolated Docker daemon**, separate from the one on your host machine. If you just run `docker build` normally, the image lands in your host's Docker, and minikube's cluster has no idea it exists. 
`eval $(minikube docker-env) or minikube docker-env | Invoke-Expression` temporarily points your terminal's `docker` commands at minikube's daemon instead of your host's, so the image you build actually ends up somewhere the cluster can see and run it. 
>This is a common source of confusion later: if you rebuild an image in a new terminal without re-running this line first, Kubernetes will keep running the old image, because as far as it's concerned nothing changed.

### Step 2: Deploy to Kubernetes

Two manifest files in **Week6/k8s**:
- [`k8s/deployment.yaml`](k8s/deployment.yaml): describes desired state, 2 replicas of our Flask container.
- [`k8s/service.yaml`](k8s/service.yaml): exposes those Pods on a stable NodePort.

```bash
kubectl apply -f k8s/deployment.yaml
kubectl apply -f k8s/service.yaml
```

**Why two separate files instead of one:** they manage two different kinds of problem, and Kubernetes treats them as independent resources on purpose. 
- The Deployment's job is "keep 2 healthy Pods running." It knows nothing about how traffic reaches them. 
- The Service's job is "give whatever Pods currently exist one stable address." It knows nothing about how many Pods there should be. 
- Pods get replaced constantly (crashes, rolling updates, scaling), and each replacement gets a **new internal IP address**. 
- Without a Service, anything trying to reach your app would need to track those changing IPs itself. The Service solves that by giving you one fixed endpoint that always routes to whichever Pods currently match its label selector. 
- The Deployment and the Service stay in sync through labels, not through being defined in the same file.

Check everything came up:
```bash
kubectl get pods
kubectl get deployments
kubectl get services
```
> Expect **2** Pods `Running`,  one `flask-service` of type `NodePort`, and `kubernetes` of type `ClusterIP`.

Reach the app:
```bash
minikube service flask-service --url
```
> Prints a URL. Open it in your browser and you should see the same JSON response as Week 5's `/` route, now served by a Kubernetes-managed Pod instead of a container you started by hand.

### Step 3: See self-healing in action

```bash
kubectl get pods                       # note one of the Pod names
kubectl delete pod <that-pod-name>     # simulate a crash
kubectl get pods                       # run again immediately after
```
> A replacement Pod should already be `ContainerCreating` or `Running`.

**Why this works, mechanically:** deleting a Pod doesn't touch the Deployment's `replicas: 2` instruction. That desired state is unchanged. 
- The moment the Pod disappears, the control loop (from the "Why Pods and Deployments" section above) notices actual state has dropped to 1 and immediately schedules a new Pod to close the gap. Nobody ran a "restart" command. 
- The same mechanism that would recover from a real crash is what you just triggered on purpose. 
> This is the concrete payoff of everything above: `restart: unless-stopped` in Docker Compose only restarts a container on the same machine, while a Kubernetes Deployment will reschedule a replacement even on a different node in a real multi-node cluster.

**Questions** [Don't forget to answer these reflection questions]
- In `k8s/deployment.yaml`, what does `replicas: 2` actually guarantee, and what happens the moment you manually delete one of those Pods?
- What is the difference between what `kubectl apply -f k8s/deployment.yaml` manages and what `kubectl apply -f k8s/service.yaml` manages? Why are they separate files/resources instead of one?
- The `readinessProbe` in `deployment.yaml` points at `/health`. What would you expect to happen to traffic routing if `/health` started returning a non-200 status?

## **7. GPU Scheduling in Kubernetes** (Self-learning @home)

**Concepts:**
- By default, Kubernetes schedules Pods based on **CPU** and **memory** requests. GPUs are exposed as an additional schedulable resource, but only once a node has:
  1. A physical/virtual GPU attached, and
  2. The **NVIDIA device plugin** running on that node, which advertises the GPU to Kubernetes as `nvidia.com/gpu`.
- Once advertised, a Pod requests a GPU the exact same way it requests CPU/memory  &rarr;  via `resources.limits`.
- **Node selectors / taints & tolerations** are used to keep GPU-hungry Pods off plain CPU nodes (and vice versa) in a mixed cluster  &rarr;  important because GPU nodes are expensive, so you don't want a random CPU-only job accidentally landing on one.
- In practice, most data science teams don't run bare-metal GPU nodes themselves  &rarr;  they use a **managed** Kubernetes service's GPU node pools (e.g. GKE Autopilot with GPUs, EKS with `g4dn` instances), which handles driver installation for you.

### Instructions (read-through, not execution)
> This section is **conceptual**  &rarr;  scheduling a real GPU Pod requires an actual GPU-equipped cluster node, which a local `minikube` cluster typically doesn't have. Treat this as manifest literacy, the same way you read `.gitlab-ci.yml` without running it.

1. Open [`k8s/gpu-pod.yaml`](k8s/gpu-pod.yaml) in the **Week6/k8s directory**.
2. Compare it to [`k8s/deployment.yaml`](k8s/deployment.yaml) from the previous section  &rarr;  find the two new pieces: the `resources.limits.nvidia.com/gpu` line, and the `nodeSelector`.
3. If your team's cluster is a managed cloud Kubernetes service, this is very close to what a training or batch-inference job manifest would actually look like  &rarr;  just with your own training image instead of the `nvidia/cuda` base image shown here.

**Questions** [Don't forget to answer these reflection questions]
- Why can't you request `nvidia.com/gpu: 0.5` the way you might request `cpu: "0.5"`? What does that imply about how GPUs are shared (or not shared) between Pods?
- What two conditions does a node need to satisfy before Kubernetes will even consider scheduling `gpu-pod.yaml` onto it?
- Why would a mixed cluster (some CPU-only nodes, some GPU nodes) use `nodeSelector` rather than just letting the scheduler pick any node with free capacity?

### Now we are done with week 6's lab content 🎉🎉🎉 ###
### Let's navigate to Exercise's notebook
[Exercise Notebook](week6_exercise.ipynb)